# Proyecto 4 - RAG sobre Seguridad Pública en México

**Stack:** Python 3.11.9, LangChain, ChromaDB, sentence-transformers (all-MiniLM-L6-v2), Ollama (Qwen2.5-7B-Instruct)

**Pipeline:**
1. **Scraper** — descarga PDFs de fuentes institucionales (23 PDFs, 13 categorías)
2. **Chunking** — extrae texto y divide en chunks de ~1000 caracteres
3. **Embeddings** — genera vectores 384d con all-MiniLM-L6-v2 y almacena en ChromaDB
4. **RAG** — recupera chunks relevantes y genera respuestas con Ollama


## 0. Setup

Asegúrate de que el entorno virtual esté activo y las dependencias instaladas.

In [ ]:
import sys, os, json, re
from pathlib import Path

# Verificar que estamos en el directorio correcto
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "4-RAG":
    PROJECT_ROOT = PROJECT_ROOT / "4-RAG"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working dir: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")


In [ ]:
# Instalar dependencias (si no están)
# !pip install -r requirements.txt


## 1. Corpus — inspección

Los PDFs ya fueron descargados por `scraper.py`. Veamos el estado actual.

In [ ]:
# Listar categorías y cantidad de PDFs
metadata_file = PROJECT_ROOT / "corpus" / "pdf" / "metadata" / "documentos.json"
with open(metadata_file) as f:
    docs = json.load(f)

print(f"Total de documentos: {len(docs)}\n")

by_cat = {}
for d in docs:
    by_cat.setdefault(d["categoria"], []).append(d)

for cat in sorted(by_cat):
    items = by_cat[cat]
    bar = "█" * len(items)
    print(f"  {cat:<30} {len(items):>2}  {bar}")

print()
for d in docs:
    print(f"  {d['archivo']:<55} {d['numero_paginas']:>4}p  ({d['institucion']})")


## 2. Chunking — extracción y división de PDFs

`src/ingest/chunking.py`:
- Carga cada PDF con **PyPDFLoader** (LangChain)
- Divide el texto con **RecursiveCharacterTextSplitter** (chunk_size=1000, overlap=200)
- Cada chunk preserva metadata: categoría, documento, páginas
- Salida: `corpus/processed/chunks.jsonl`

In [ ]:
# Ejecutar chunking
!python src/ingest/chunking.py


In [ ]:
# Explorar los chunks generados
chunks_file = PROJECT_ROOT / "corpus" / "processed" / "chunks.jsonl"
chunks = []
with open(chunks_file) as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Total chunks: {len(chunks)}\n")

# Estadísticas por categoría
from collections import Counter
cat_counts = Counter(c["metadata"]["category"] for c in chunks)
for cat, count in sorted(cat_counts.items()):
    bar = "█" * (count // 50)
    print(f"  {cat:<30} {count:>5}  {bar}")

print(f"\nEjemplo de chunk:")
c = chunks[0]
print(f"  ID: {c['chunk_id']}")
print(f"  Categoría: {c['metadata']['category']}")
print(f"  Páginas: {c['metadata']['pages']}")
print(f"  Texto: {c['text'][:150]}...")


## 3. Embeddings — vectorización con all-MiniLM-L6-v2

`src/embeddings/embeddings.py`:
- Modelo: `sentence-transformers/all-MiniLM-L6-v2` (384 dimensiones, CPU)
- Almacenamiento: **ChromaDB** persistente en `vectordb/`
- Cada chunk se almacena con su vector + metadata

In [ ]:
# Indexar en ChromaDB (--rebuild fuerza reindexación)
!python src/embeddings/embeddings.py --rebuild


In [ ]:
# Verificar el vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

vectordb = Chroma(
    persist_directory=str(PROJECT_ROOT / "vectordb"),
    embedding_function=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
    ),
    collection_name="seguridad_mexico",
)

count = vectordb._collection.count()
print(f"Documentos indexados: {count}")

# Test de búsqueda semántica
results = vectordb.similarity_search("homicidios por estado", k=3)
print(f"\nResultados de búsqueda:\n")
for r in results:
    cat = r.metadata.get("category", "?")
    src = r.metadata.get("source_file", "?")
    pages = r.metadata.get("pages", "?")
    print(f"  [{cat}] {src} (págs: {pages})")
    print(f"  {r.page_content[:120]}...\n")


## 4. RAG — consultas con recuperación + generación

`src/rag/rag.py`:
- **Retriever**: top-k=5 chunks más similares desde ChromaDB
- **LLM**: `Qwen2.5-7B-Instruct` vía Ollama
- **Prompt**: template en español, instruye a usar solo el contexto
- **Chain**: LCEL (LangChain Expression Language)

In [ ]:
# Consulta directa (sin Ollama — solo recuperación)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

vectordb = Chroma(
    persist_directory=str(PROJECT_ROOT / "vectordb"),
    embedding_function=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
    ),
    collection_name="seguridad_mexico",
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
results = retriever.invoke("¿Qué relación hay entre violencia y pobreza?")

print("Top 3 chunks recuperados:\n")
for i, r in enumerate(results):
    cat = r.metadata.get("category", "?")
    src = r.metadata.get("source_file", "?")
    print(f"[{i+1}] {cat} — {src}")
    print(f"    {r.page_content[:200]}...\n")


### 4.1 RAG completo (con Ollama)

Para usar el RAG completo, necesitas **Ollama** instalado y el modelo descargado:

In [ ]:
# Descargar modelo Ollama (una vez)
!ollama pull qwen2.5:7b-instruct


Luego probamos el RAG con una pregunta:

In [ ]:
# Consulta RAG completa
!python src/rag/rag.py "¿Qué estados de México tienen las tasas más altas de homicidios?"


## 5. Pipeline completo

Para ejecutar todo el pipeline de una sola vez:

In [ ]:
# Pipeline completo (chunking + embeddings)
!python run_pipeline.py


In [ ]:
# Pipeline + RAG interactivo
# !python run_pipeline.py --rag


## Resumen

| Paso | Script | Output |
|---|---|---|
| 1. Scraper | `src/ingest/scraper.py` | 23 PDFs en `corpus/pdf/` |
| 2. Chunking | `src/ingest/chunking.py` | 7,595 chunks en `corpus/processed/chunks.jsonl` |
| 3. Embeddings | `src/embeddings/embeddings.py` | ChromaDB en `vectordb/` (384d, cosine) |
| 4. RAG | `src/rag/rag.py` | Retrieval + Qwen2.5-7B vía Ollama |

**Arquitectura:**
```
Pregunta → Retriever (ChromaDB, top-5) → Contexto → LLM (Ollama) → Respuesta
```
